# Lab 07 — AI_REDACT & AI_PARSE_DOCUMENT

**Redact** masks PII automatically. **Parse Document** extracts text from PDFs and images.

| Function | Returns | Use Case |
|---|---|---|
| `AI_REDACT` | Text with PII masked | GDPR/CCPA compliance, data sharing |
| `AI_PARSE_DOCUMENT` | Extracted text/structure from files | Invoice processing, contract analysis |

**What You'll Learn**

- `AI_REDACT` — all categories, selective PII, and detect mode
- OCR vs LAYOUT document parsing modes with `AI_PARSE_DOCUMENT`
- Page-split extraction for multi-page documents
- Chaining: parse → extract with `AI_COMPLETE`

## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
Equivalent legacy functions: `AI_COMPLETE` (replaces `SNOWFLAKE.CORTEX.COMPLETE`), `AI_PARSE_DOCUMENT` (replaces `SNOWFLAKE.CORTEX.PARSE_DOCUMENT`).
 `AI_REDACT` — new, no legacy equivalent.

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

-- Database pre-created by SYSADMIN in 00-admin-setup
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — AI_REDACT: Automatic PII Detection

In [ ]:
CREATE OR REPLACE TABLE pii_records (
    record_id INT AUTOINCREMENT, record_type VARCHAR(50), raw_text TEXT
);

INSERT INTO pii_records (record_type, raw_text) VALUES
('support_ticket', 'Hi, my name is Sarah Johnson and my account number is ACC-789456. I can be reached at sarah.johnson@email.com or 555-0142. My shipping address is 742 Evergreen Terrace, Springfield, IL 62704.'),
('medical_note', 'Patient: Robert Chen, DOB: 03/15/1985, SSN: 321-54-9876. Presented with symptoms of seasonal allergies. Prescribed cetirizine 10mg daily.'),
('financial_record', 'Transfer from James Williams (account 4532-1234-5678-9012) to Maria Garcia (account 5678-9012-3456-7890). Amount: $15,750.00.'),
('employee_record', 'New hire: Emily Davis, Employee ID: EMP-2024-0892. Personal email: emily.d@gmail.com. Phone: (408) 555-0198.'),
('customer_chat', 'My username is jdoe2024 and my registered phone is 212-555-0167 and the email on file should be john.doe@company.com.');

In [ ]:
SELECT
    record_type,
    AI_REDACT(raw_text) AS redacted_text
FROM pii_records;

In [ ]:
SELECT
    record_type,
    LEFT(raw_text, 100) AS original,
    LEFT(AI_REDACT(raw_text)::STRING, 100) AS redacted
FROM pii_records;

## Step 2b — AI_REDACT: Selective Categories & Detect Mode

AI_REDACT supports two important options:
- **Categories array** — redact only specific PII types (e.g. only names and emails, leave addresses visible)
- **Detect mode** — returns PII span locations *without* redacting, useful for auditing

In [ ]:
-- Redact only names and emails (leave phone numbers, addresses visible)
SELECT
    record_type,
    raw_text,
    AI_REDACT(raw_text, ['NAME', 'EMAIL']) AS names_emails_redacted
FROM pii_records
LIMIT 3;

In [ ]:
-- Detect mode: returns PII spans without redacting
SELECT
    record_type,
    LEFT(raw_text, 80) AS text_preview,
    AI_REDACT(input => raw_text, mode => 'detect') AS detected_pii
FROM pii_records
LIMIT 3;

## Step 3 — AI_PARSE_DOCUMENT: Extract from PDFs

`AI_PARSE_DOCUMENT` extracts text from documents on a Snowflake stage.
Two modes: `OCR` (raw text) and `LAYOUT` (preserves structure).


In [ ]:
CREATE OR REPLACE STAGE invoice_pdfs
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
    COMMENT = 'Invoice PDF files for AI_PARSE_DOCUMENT demo';

CREATE OR REPLACE STAGE lab_images
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
    COMMENT = 'Image files for AI_COMPLETE vision demos';

### Generate Sample Invoice PDFs

The cell below creates two simple invoice PDFs and uploads them to `@invoice_pdfs`.

In [ ]:
import io
from snowflake.snowpark.context import get_active_session

session = get_active_session()

def make_pdf(lines):
    """Build a minimal valid PDF from a list of text lines."""
    stream_objects = []
    page_content = ""
    y = 750
    for line in lines:
        page_content += f"BT /F1 11 Tf {50} {y} Td ({line}) Tj ET\n"
        y -= 16

    stream = page_content.encode()
    objects = [
        b"%PDF-1.4\n",
        # 1: Catalog
        b"1 0 obj<</Type/Catalog/Pages 2 0 R>>endobj\n",
        # 2: Pages
        b"2 0 obj<</Type/Pages/Kids[3 0 R]/Count 1>>endobj\n",
        # 3: Page
        b"3 0 obj<</Type/Page/Parent 2 0 R/MediaBox[0 0 612 792]/Contents 4 0 R/Resources<</Font<</F1 5 0 R>>>>>>endobj\n",
        # 4: Stream
        b"4 0 obj<</Length " + str(len(stream)).encode() + b">>stream\n" + stream + b"\nendstream\nendobj\n",
        # 5: Font
        b"5 0 obj<</Type/Font/Subtype/Type1/BaseFont/Helvetica>>endobj\n",
    ]
    body = b"".join(objects)
    xref_offset = len(body)
    xref = b"xref\n0 6\n"
    xref += b"0000000000 65535 f \n"
    offset = len(objects[0])
    for obj in objects[1:]:
        xref += f"{offset:010d} 00000 n \n".encode()
        offset += len(obj)
    trailer = f"trailer<</Size 6/Root 1 0 R>>\nstartxref\n{xref_offset}\n%%EOF".encode()
    return body + xref + trailer

invoices = {
    "invoice_001.pdf": [
        "INVOICE #INV-2026-001",
        "Date: 2026-01-15",
        "Vendor: Acme Cloud Services Inc.",
        "Bill To: Snowflake Corp",
        "",
        "Description                  Qty    Unit Price    Amount",
        "-----------------------------------------------------------",
        "Compute Credits (Standard)   500    $3.00         $1,500.00",
        "Storage (TB/month)           10     $23.00        $230.00",
        "Data Transfer (TB)           5      $20.00        $100.00",
        "",
        "Subtotal:                                         $1,830.00",
        "Tax (8%):                                         $146.40",
        "TOTAL DUE:                                        $1,976.40",
        "",
        "Payment Terms: Net 30",
        "Due Date: 2026-02-14",
    ],
    "invoice_002.pdf": [
        "INVOICE #INV-2026-002",
        "Date: 2026-02-20",
        "Vendor: DataStream Analytics LLC",
        "Bill To: Snowflake Corp",
        "",
        "Description                  Qty    Unit Price    Amount",
        "-----------------------------------------------------------",
        "ML Model Training (hours)    200    $5.50         $1,100.00",
        "GPU Compute (A100, hours)    80     $12.00        $960.00",
        "API Calls (millions)         15     $0.50         $7.50",
        "",
        "Subtotal:                                         $2,067.50",
        "Tax (8%):                                         $165.40",
        "TOTAL DUE:                                        $2,232.90",
        "",
        "Payment Terms: Net 45",
        "Due Date: 2026-04-05",
    ],
}

for filename, lines in invoices.items():
    pdf_bytes = make_pdf(lines)
    buf = io.BytesIO(pdf_bytes)
    session.file.put_stream(buf, f"@invoice_pdfs/{filename}", auto_compress=False, overwrite=True)
    print(f"Uploaded {filename} ({len(pdf_bytes):,} bytes)")

session.sql("ALTER STAGE invoice_pdfs REFRESH").collect()
print("Stage refreshed")

In [ ]:
LIST @invoice_pdfs;

## Step 3b — AI_PARSE_DOCUMENT: Page Split Mode

Set `page_split: true` to process each page independently. The output includes a `pages` array with per-page `content` and `index`. This is essential for long documents that exceed the token limit.

In [ ]:
SELECT
    RELATIVE_PATH AS file_name,
    AI_PARSE_DOCUMENT(
        TO_FILE('@invoice_pdfs', RELATIVE_PATH),
        {'mode': 'LAYOUT', 'page_split': true}
    ):pages AS pages
FROM DIRECTORY(@invoice_pdfs)
LIMIT 1;

In [ ]:
SELECT
    RELATIVE_PATH AS file_name,
    AI_PARSE_DOCUMENT(
        TO_FILE('@invoice_pdfs', RELATIVE_PATH),
        {'mode': 'OCR'}
    ) AS parsed_content
FROM DIRECTORY(@invoice_pdfs)
LIMIT 3;

In [ ]:
SELECT
    RELATIVE_PATH AS file_name,
    AI_PARSE_DOCUMENT(
        TO_FILE('@invoice_pdfs', RELATIVE_PATH),
        {'mode': 'LAYOUT'}
    ) AS parsed_content
FROM DIRECTORY(@invoice_pdfs)
LIMIT 3;

## Step 4 — Chain: Parse → Extract with COMPLETE

In [ ]:
WITH parsed AS (
    SELECT RELATIVE_PATH AS file_name,
        AI_PARSE_DOCUMENT(
            TO_FILE('@invoice_pdfs', RELATIVE_PATH), {'mode': 'LAYOUT'}
        ):content::STRING AS doc_text
    FROM DIRECTORY(@invoice_pdfs)
)
SELECT file_name,
    AI_COMPLETE('claude-haiku-4-5',
        'Extract from this invoice and return JSON: {invoice_number, date, vendor_name, total_amount}. Document: ' || doc_text
    ) AS extracted_fields
FROM parsed;

## Key Takeaways

- `AI_REDACT` detects names, SSNs, emails, phones, addresses — no configuration needed.
- `AI_PARSE_DOCUMENT` supports `OCR` (raw text) and `LAYOUT` (structure-preserving) modes.
- Chain `AI_PARSE_DOCUMENT` → `AI_COMPLETE` to extract specific fields from any document.
- Essential for compliance (GDPR/CCPA), data sharing, and document processing pipelines.
